# MP37: feature fusion

We compare aligned NCHW feature maps, using CPU float32. Sum and mean preserve C channels; concatenation preserves both branches as2C channels before a learned1x1 projection. These are teaching tensors, not sensor observations. See [lesson](README.md) for assumptions, diagram and exercises.

In [ ]:
import torch
from mp37_tools import fuse_features, projection_cost, initialize_mean_projection
torch.manual_seed(17)
torch.set_num_threads(2)
a=torch.tensor([1.,2.]).reshape(1,2,1,1).requires_grad_()
b=torch.tensor([3.,4.]).reshape(1,2,1,1).requires_grad_()
print("sum",fuse_features(a,b,"sum").flatten().tolist())
print("mean",fuse_features(a,b).flatten().tolist())
print("concat shape",tuple(fuse_features(a,b,"concat").shape))

## Projection and gradients

P=[I/2,I/2] with zero bias exactly represents averaging. dM/dA=dM/dB=1/2. This equality does not imply that unconstrained training produces the same function.

In [ ]:
p=initialize_mean_projection(torch.nn.Conv2d(4,2,1))
y=p(fuse_features(a,b,"concat"))
torch.testing.assert_close(y,fuse_features(a,b))
y.sum().backward()
print("projected",y.detach().flatten().tolist(),"gradients",a.grad.flatten().tolist(),b.grad.flatten().tolist())
assert projection_cost(8,5,7)=={"parameters":136,"macs":4480}

## Controlled shared-model fixture and reused evidence

The next cell copies common weights between mean and concat models, sets the projector to the exact mean, and checks an odd rectangular input. Historical scores come from existing validation runs and are kept separate; no training or test evaluation runs here.

In [ ]:
from experiment import run
result=run()
print(result["model_fixture"])
for name,row in result["historical"].items():
    print(name,row["psnr_mean_db"],row["sample_seed_sd_db"],row["parameters"])
print(result["limitations"])

## Exercise and answer

For2x4x3x5 features, concat has2x8x3x5 elements; a2C->C projection has36 parameters and960 MACs. Why not call the larger historical score proof of better architecture? Because the one-scene pilot has differing parameter budgets and seed variation is not geographic uncertainty. A1x1 projection mixes channels at a pixel but cannot itself shift an incorrectly registered image.